# OPD No-Show Analysis — Capstone Notebook
**AI-Driven Analysis of OPD Appointment No-Shows for Revenue Optimization**

### Setup Instructions
1. Download the dataset from Kaggle: https://www.kaggle.com/datasets/joniarroba/noshowappointments
2. Place `KaggleV2-May-2016.csv` in the **same folder** as this notebook
3. Install dependencies: `pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost shap statsmodels fairlearn`
4. Run cells top-to-bottom — every cell depends on the ones above it

---
## Cell 1 — Install Dependencies

In [ ]:
# Run this cell once to install all required packages
import subprocess, sys

packages = [
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'scikit-learn', 'imbalanced-learn', 'xgboost',
    'shap', 'statsmodels', 'fairlearn'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages installed successfully.')

---
## Cell 2 — Load Dataset & Preprocessing Pipeline

**Steps:**
1. Load CSV from local path
2. Remove 5 invalid-age records (Age < 0)
3. Engineer `WaitDays` feature
4. Recode target variable (`No-show` → binary `NoShow`)
5. Encode categorical features
6. Create `WaitBucket` categorical groups
7. Train/test split (80/20 stratified) + SMOTE on training set only

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG: update path if dataset is in a different folder ──────────────────
DATASET_PATH = r'C:\Users\DELL\Downloads\DATA ANALYTICS CAPSTONE\KaggleV2-May-2016.csv' # CSV or .zip both accepted
# ────────────────────────────────────────────────────────────────────────────

# Load (handles both .csv and .zip automatically)
if DATASET_PATH.endswith('.zip'):
    df = pd.read_csv(DATASET_PATH, compression='zip')
else:
    df = pd.read_csv(DATASET_PATH)

print(f'Raw shape: {df.shape}')          # Expected: (110527, 14)
print(df.head(2))

# ── Step 1: Remove invalid age records ──────────────────────────────────────
df = df[df['Age'] >= 0].copy()
print(f'After age filter: {df.shape}')   # Expected: (110522, 14)

# ── Step 2: Engineer WaitDays ────────────────────────────────────────────────
df['ScheduledDay']   = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['WaitDays'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days.clip(lower=0)
print(f'WaitDays — min: {df["WaitDays"].min()}, max: {df["WaitDays"].max()}, '
      f'median: {df["WaitDays"].median()}, mean: {df["WaitDays"].mean():.1f}')

# ── Step 3: Recode target ("Yes" = patient did NOT attend = 1) ───────────────
df['NoShow'] = (df['No-show'] == 'Yes').astype(int)
print('\nTarget distribution:')
print(df['NoShow'].value_counts(normalize=True).round(4))
# Expected: 0 (Show): 0.7981  |  1 (No-Show): 0.2019

# ── Step 4: Encode categoricals ─────────────────────────────────────────────
df['Gender_enc'] = (df['Gender'] == 'M').astype(int)   # F=0, M=1
from sklearn.preprocessing import LabelEncoder
df['Nbhd_enc'] = LabelEncoder().fit_transform(df['Neighbourhood'])

# ── Step 5: WaitDays buckets ─────────────────────────────────────────────────
bins   = [-1, 0, 7, 15, 30, 999]
labels = ['0', '1-7', '8-15', '16-30', '31+']
df['WaitBucket'] = pd.cut(df['WaitDays'], bins=bins, labels=labels)
print('\nNo-show rate by WaitBucket:')
print(df.groupby('WaitBucket')['NoShow'].mean().round(4))

# ── Step 6: Train / test split ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split

FEATURES = [
    'Gender_enc', 'Age', 'Scholarship', 'Hipertension',
    'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received',
    'WaitDays', 'Nbhd_enc'
]
X = df[FEATURES]
y = df['NoShow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f'\nTrain size: {X_train.shape[0]:,}  |  Test size: {X_test.shape[0]:,}')
print(f'Train positive rate: {y_train.mean():.4f}  |  Test positive rate: {y_test.mean():.4f}')

# ── Step 7: SMOTE on training set ONLY (prevents data leakage) ───────────────
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print(f'\nAfter SMOTE — shape: {X_res.shape}  |  positive rate: {y_res.mean():.4f}')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\DELL\\Downloads\\DATA ANALYTICS CAPSTONE\\KaggleV2-May-2016.csv'

---
## Cell 3 — EDA: Class Distribution (Figure 5.1)

Visualise the 80/20 class imbalance that motivates SMOTE and AUC-first evaluation.

In [ ]:
BLUE='#2E75B6'; LBLUE='#BDD7EE'; RED='#C00000'
BASELINE = df['NoShow'].mean()

shows   = (df['NoShow'] == 0).sum()
noshows = (df['NoShow'] == 1).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Figure 5.1 — Target Variable Distribution and Class Imbalance',
             fontsize=13, fontweight='bold', color=BLUE, y=1.01)

bars = ax1.bar(['Attended (Show)', 'No-Show'], [shows, noshows],
               color=[BLUE, RED], width=0.5, edgecolor='white')
ax1.bar_label(bars, labels=[f'{shows:,}', f'{noshows:,}'],
              fontsize=12, fontweight='bold', padding=4)
ax1.set_ylabel('Number of Appointments', fontsize=10)
ax1.set_title('Appointment Counts by Class', fontsize=11, color=BLUE)
ax1.set_ylim(0, shows * 1.15)
ax1.spines[['top', 'right']].set_visible(False)

wedges, texts, autotexts = ax2.pie(
    [shows, noshows],
    labels=[f'Show\n({shows/len(df)*100:.2f}%)', f'No-Show\n({noshows/len(df)*100:.2f}%)'],
    colors=[LBLUE, RED], autopct='%1.2f%%', startangle=90,
    textprops={'fontsize': 10},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
ax2.set_title('Class Proportion', fontsize=11, color=BLUE)

fig.text(0.5, -0.04,
    f'Insight: The {BASELINE*100:.2f}% no-show rate represents {noshows:,} missed appointments — '
     'each a direct revenue loss opportunity. Imbalance drives model evaluation choices.',
    ha='center', va='top', fontsize=9, style='italic', color='#1F3864',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#D9E8FB', edgecolor='#2E75B6'))

plt.tight_layout()
plt.savefig('fig5_1_class_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'No-show rate: {BASELINE:.4f} ({noshows:,} / {len(df):,})')

---
## Cell 4 — EDA: No-Show Rate by Age Group (Figure 5.2)

In [ ]:
import matplotlib.patches as mpatches

ORANGE = '#E36C09'

# Compute ACTUAL rates from data
age_bins   = [-1, 12, 17, 30, 45, 60, 75, 120]
age_labels = ['0-12\n(Child)', '13-17\n(Teen)', '18-30\n(Young\nAdult)',
              '31-45\n(Adult)', '46-60\n(Mid-age)', '61-75\n(Senior)', '76+\n(Elderly)']
df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)
age_rates = df.groupby('AgeGroup', observed=True)['NoShow'].mean()

print('Actual no-show rate by age group:')
for grp, rate in age_rates.items():
    marker = '▲' if rate > BASELINE else '▼'
    print(f'  {str(grp):25s}: {rate*100:.1f}%  {marker}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Figure 5.2 — No-Show Rate by Age Group',
             fontsize=13, fontweight='bold', color=BLUE, y=1.01)

colors = [RED if r > BASELINE else LBLUE for r in age_rates.values]
bars = ax1.bar(age_labels, age_rates.values * 100, color=colors, edgecolor='white')
ax1.bar_label(bars, labels=[f'{r*100:.1f}%' for r in age_rates.values],
              fontsize=9, fontweight='bold', padding=3)
ax1.axhline(BASELINE * 100, color=ORANGE, linewidth=2, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax1.set_ylabel('No-Show Rate (%)', fontsize=10)
ax1.set_title('No-Show Rate by Age Group', fontsize=11, color=BLUE)
ax1.set_ylim(0, max(age_rates.values) * 100 * 1.2)
ax1.spines[['top', 'right']].set_visible(False)
above = mpatches.Patch(color=RED,   label=f'Above baseline ({BASELINE*100:.2f}%)')
below = mpatches.Patch(color=LBLUE, label='Below baseline')
ax1.legend(handles=[above, below], fontsize=8.5, loc='upper right')

# Continuous age trend (actual data)
age_trend = df.groupby('Age')['NoShow'].mean()
ax2.plot(age_trend.index, age_trend.values * 100, color='#1F3864', linewidth=2.5)
ax2.axhline(BASELINE * 100, color=ORANGE, linewidth=1.8, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax2.axvspan(18, 30, alpha=0.15, color=RED, label='Peak risk zone')
ax2.set_xlabel('Patient Age (years)', fontsize=10)
ax2.set_ylabel('No-Show Rate (%)', fontsize=10)
ax2.set_title('Continuous Age vs. No-Show Rate', fontsize=11, color=BLUE)
ax2.legend(fontsize=8.5)
ax2.set_xlim(0, 100)
ax2.spines[['top', 'right']].set_visible(False)

young_rate = df[df['Age'].between(18, 30)]['NoShow'].mean()
elderly_rate = df[df['Age'] >= 76]['NoShow'].mean()
fig.text(0.5, -0.04,
    f'Insight: Young adults (18–30) have the highest no-show rate ({young_rate*100:.1f}%), '
    f'{young_rate/BASELINE:.0%} above baseline. '
    f'Elderly patients (76+) are the most reliable attenders ({elderly_rate*100:.1f}%).',
    ha='center', va='top', fontsize=9, style='italic', color='#1F3864',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#D9E8FB', edgecolor='#2E75B6'))

plt.tight_layout()
plt.savefig('fig5_2_age_group.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 5 — EDA: No-Show Rate by WaitDays (Figure 5.3)

Kruskal-Wallis test + Decision Tree threshold + trend chart.

In [ ]:
from scipy.stats import kruskal
from sklearn.tree import DecisionTreeClassifier, export_text

LGREEN = '#A9D18E'; GREEN = '#375623'

# ── Kruskal-Wallis test ──────────────────────────────────────────────────────
groups = [df[df['WaitBucket'] == b]['NoShow'].values for b in labels]
stat, p = kruskal(*groups)
print(f'Kruskal-Wallis H={stat:.2f}, p={p:.2e}')
print(f'Decision: {"Reject H₀ — WaitDays significantly affects no-show" if p < 0.05 else "Fail to reject H₀"}')

# ── Actual bucket rates ──────────────────────────────────────────────────────
rates = df.groupby('WaitBucket', observed=True)['NoShow'].mean().reset_index()
rates.columns = ['WaitBucket', 'NoShowRate']
print('\nActual no-show rate by WaitDays bucket:')
print(rates.to_string(index=False))

# ── Decision Tree: natural threshold ─────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(df[['WaitDays']], df['NoShow'])
print('\nDecision Tree splits on WaitDays:')
print(export_text(dt, feature_names=['WaitDays']))

# ── Figure 5.3 ───────────────────────────────────────────────────────────────
bucket_vals  = rates['NoShowRate'].values
bucket_labs  = rates['WaitBucket'].astype(str).tolist()
bucket_colors = []
for r in bucket_vals:
    if r < BASELINE: bucket_colors.append(LGREEN)
    elif r < 0.25:   bucket_colors.append('#E36C09')
    else:            bucket_colors.append(RED)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Figure 5.3 — No-Show Rate by Scheduling Lead Time (WaitDays)',
             fontsize=13, fontweight='bold', color=BLUE, y=1.01)

bars = ax1.bar(bucket_labs, bucket_vals * 100, color=bucket_colors, edgecolor='white')
ax1.bar_label(bars, labels=[f'{r*100:.1f}%' for r in bucket_vals],
              fontsize=10, fontweight='bold', padding=3)
ax1.axhline(BASELINE * 100, color=BLUE, linewidth=2, linestyle='--',
            label=f'Dataset mean ({BASELINE*100:.2f}%)')
ax1.set_ylabel('No-Show Rate (%)', fontsize=10)
ax1.set_title('No-Show Rate by WaitDays Bucket', fontsize=11, color=BLUE)
ax1.set_ylim(0, max(bucket_vals) * 100 * 1.2)
ax1.legend(fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)

# Actual trend 0-60 days
trend = df[df['WaitDays'] <= 60].groupby('WaitDays')['NoShow'].mean()
ax2.plot(trend.index, trend.values * 100, color=RED, linewidth=2.5,
         label='No-show rate trend')
ax2.axhline(BASELINE * 100, color=BLUE, linewidth=1.5, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax2.axvline(7,  color=ORANGE, linewidth=1.5, linestyle='--', label='7-day mark')
ax2.axvline(15, color=RED,    linewidth=1.5, linestyle=':',  label='15-day policy threshold')
ax2.fill_between(trend.index, trend.values * 100, BASELINE * 100,
                 where=trend.values > BASELINE, alpha=0.12, color=RED)
ax2.set_xlabel('WaitDays (days between booking and appointment)', fontsize=10)
ax2.set_ylabel('No-Show Rate (%)', fontsize=10)
ax2.set_title('No-Show Rate Trend (0–60 days)', fontsize=11, color=BLUE)
ax2.set_ylim(0, 45)
ax2.legend(fontsize=8.5)
ax2.spines[['top', 'right']].set_visible(False)

same_day = df[df['WaitDays'] == 0]['NoShow'].mean()
long_wait = df[df['WaitDays'] >= 31]['NoShow'].mean()
fig.text(0.5, -0.04,
    f'Insight: No-show rate rises {long_wait/same_day:.0f}× from same-day ({same_day*100:.1f}%) '
    f'to 31+ days ({long_wait*100:.1f}%). Kruskal-Wallis p={p:.2e} confirms significance.',
    ha='center', va='top', fontsize=9, style='italic', color='#1F3864',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#D9E8FB', edgecolor='#2E75B6'))

plt.tight_layout()
plt.savefig('fig5_3_waitdays.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 6 — EDA: SMS Reminder Paradox (Figure 5.4)

Chi-Square test + interaction logistic regression + stratified subgroup rates.

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

# ── Raw Chi-Square ───────────────────────────────────────────────────────────
ct = pd.crosstab(df['SMS_received'], df['NoShow'])
chi2_sms, p_sms, dof, _ = chi2_contingency(ct)
print(f'SMS Chi-Square: chi2={chi2_sms:.2f}, p={p_sms:.4f}')

sms_raw_rates = df.groupby('SMS_received')['NoShow'].mean()
print(f'No SMS rate: {sms_raw_rates[0]*100:.1f}%  |  SMS rate: {sms_raw_rates[1]*100:.1f}%')
print('(Counter-intuitive — explained by confounding with WaitDays)')

# ── Interaction logistic regression ─────────────────────────────────────────
df['ShortWait'] = (df['WaitDays'] <= 7).astype(int)
mod = smf.logit(
    'NoShow ~ SMS_received + WaitDays + Scholarship + Age'
    ' + SMS_received:WaitDays + SMS_received:Scholarship',
    data=df).fit(maxiter=200, disp=False)

print('\n── Interaction Regression Summary ──')
print(mod.summary().tables[1])   # coefficients table only

# ── Stratified subgroup rates ────────────────────────────────────────────────
df['Group'] = df['SMS_received'].astype(str) + '_' + df['ShortWait'].astype(str)
strat = df.groupby('Group')['NoShow'].agg(['mean', 'count'])
strat = strat.rename(columns={'mean': 'NoShowRate', 'count': 'N'})
strat['Label'] = ['Long Wait No SMS', 'Long Wait SMS', 'Short Wait No SMS', 'Short Wait SMS']
print('\nStratified No-Show Rates:')
print(strat[['Label', 'NoShowRate', 'N']].to_string(index=False))

# ── Figure 5.4 ───────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Figure 5.4 — SMS Reminder Paradox: Raw vs. Adjusted Effect',
             fontsize=13, fontweight='bold', color=BLUE, y=1.01)

# Raw (confounded)
raw_vals = [sms_raw_rates[0] * 100, sms_raw_rates[1] * 100]
bars1 = ax1.bar(['No SMS\nReceived', 'SMS\nReceived'], raw_vals,
                color=[BLUE, RED], width=0.45, edgecolor='white')
ax1.bar_label(bars1, labels=[f'{v:.1f}%' for v in raw_vals],
              fontsize=12, fontweight='bold', padding=4)
ax1.axhline(BASELINE * 100, color=ORANGE, linewidth=1.8, linestyle='--',
            label=f'Dataset mean ({BASELINE*100:.2f}%)')
ax1.set_ylabel('Raw No-Show Rate (%)', fontsize=10)
ax1.set_title('Raw SMS Effect (Confounded)', fontsize=11, color=BLUE)
ax1.set_ylim(0, max(raw_vals) * 1.25)
ax1.legend(fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)
ax1.annotate('⚠ Counter-intuitive:\nSMS → higher no-show',
             xy=(1, raw_vals[1]), xytext=(0.4, raw_vals[1] + 3),
             fontsize=8.5, color='#7B0000', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFE0E0', edgecolor=RED),
             arrowprops=dict(arrowstyle='->', color=RED))

# Adjusted (stratified)
group_map = {
    '0_0': 'Long Wait\nNo SMS',
    '0_1': 'Short Wait\nNo SMS',
    '1_0': 'Long Wait\nSMS',
    '1_1': 'Short Wait\nSMS'
}
adj_order  = ['0_1', '1_1', '0_0', '1_0']
adj_labels = ['Short Wait\nNo SMS', 'Short Wait\nSMS', 'Long Wait\nNo SMS', 'Long Wait\nSMS']
adj_vals   = [df[df['Group'] == g]['NoShow'].mean() * 100 for g in adj_order]
adj_colors = [LBLUE, LGREEN, ORANGE, RED]

bars2 = ax2.bar(adj_labels, adj_vals, color=adj_colors, edgecolor='white', width=0.55)
ax2.bar_label(bars2, labels=[f'{v:.1f}%' for v in adj_vals],
              fontsize=10, fontweight='bold', padding=3)
ax2.axhline(BASELINE * 100, color='grey', linewidth=1.5, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax2.set_ylabel('No-Show Rate (%)', fontsize=10)
ax2.set_title('Adjusted SMS Effect by Wait-Time Subgroup', fontsize=11, color=BLUE)
ax2.set_ylim(0, max(adj_vals) * 1.2)
ax2.legend(fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

fig.text(0.5, -0.04,
    'Insight: The SMS paradox is entirely a confounding artefact. After stratifying by wait-time, '
    'SMS reduces no-shows for short-wait appointments but worsens outcomes for long-wait bookings.',
    ha='center', va='top', fontsize=9, style='italic', color='#1F3864',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#D9E8FB', edgecolor='#2E75B6'))

plt.tight_layout()
plt.savefig('fig5_4_sms_paradox.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 7 — EDA: Chronic Conditions & Scholarship (Figure 5.5)

In [ ]:
# ── Actual rates from data ────────────────────────────────────────────────────
cond_labels = ['Overall\nBaseline', 'Hypertension\n(Yes)', 'Diabetes\n(Yes)',
               'Alcoholism\n(Yes)', 'Disability\n(Any)']
cond_rates = [
    BASELINE,
    df[df['Hipertension'] == 1]['NoShow'].mean(),
    df[df['Diabetes']     == 1]['NoShow'].mean(),
    df[df['Alcoholism']   == 1]['NoShow'].mean(),
    df[df['Handcap']      >= 1]['NoShow'].mean(),
]
print('No-show rates by chronic condition:')
for lbl, r in zip(cond_labels, cond_rates):
    print(f'  {lbl.replace(chr(10)," "):25s}: {r*100:.1f}%')

# Scholarship × Gender subgroups
sch_labels = ['No Scholarship\n(Male)', 'No Scholarship\n(Female)',
              'Scholarship\n(Male)', 'Scholarship\n(Female)']
sch_rates = [
    df[(df['Scholarship']==0) & (df['Gender']=='M')]['NoShow'].mean(),
    df[(df['Scholarship']==0) & (df['Gender']=='F')]['NoShow'].mean(),
    df[(df['Scholarship']==1) & (df['Gender']=='M')]['NoShow'].mean(),
    df[(df['Scholarship']==1) & (df['Gender']=='F')]['NoShow'].mean(),
]
print('\nNo-show rates by Scholarship × Gender:')
for lbl, r in zip(sch_labels, sch_rates):
    print(f'  {lbl.replace(chr(10)," "):30s}: {r*100:.1f}%')

# Chi-Square for each condition
print('\nChi-Square tests:')
for col in ['Hipertension', 'Diabetes', 'Alcoholism', 'Scholarship']:
    ct = pd.crosstab(df[col], df['NoShow'])
    chi2, p, _, _ = chi2_contingency(ct)
    print(f'  {col}: chi2={chi2:.1f}, p={p:.4f}')

# ── Figure 5.5 ───────────────────────────────────────────────────────────────
cond_colors = [LBLUE if r <= BASELINE else ORANGE for r in cond_rates]
cond_colors[0] = LBLUE  # baseline always blue
cond_colors[3] = ORANGE  # alcoholism elevated

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Figure 5.5 — No-Show Rate by Chronic Condition and Socioeconomic Status',
             fontsize=13, fontweight='bold', color=BLUE, y=1.01)

bars1 = ax1.bar(cond_labels, [r * 100 for r in cond_rates],
                color=cond_colors, edgecolor='white')
ax1.bar_label(bars1, labels=[f'{r*100:.1f}%' for r in cond_rates],
              fontsize=10, fontweight='bold', padding=3)
ax1.axhline(BASELINE * 100, color=RED, linewidth=2, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax1.set_ylabel('No-Show Rate (%)', fontsize=10)
ax1.set_title('No-Show Rate by Chronic Condition', fontsize=11, color=BLUE)
ax1.set_ylim(0, max(cond_rates) * 100 * 1.3)
ax1.legend(fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)

sch_colors = [LBLUE, LBLUE, ORANGE, RED]
bars2 = ax2.bar(sch_labels, [r * 100 for r in sch_rates],
                color=sch_colors, edgecolor='white')
ax2.bar_label(bars2, labels=[f'{r*100:.1f}%' for r in sch_rates],
              fontsize=10, fontweight='bold', padding=3)
ax2.axhline(BASELINE * 100, color='grey', linewidth=1.5, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax2.set_ylabel('No-Show Rate (%)', fontsize=10)
ax2.set_title('No-Show Rate: Scholarship × Gender Subgroups', fontsize=11, color=BLUE)
ax2.set_ylim(0, max(sch_rates) * 100 * 1.3)
ax2.legend(fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

hyp_rate = df[df['Hipertension']==1]['NoShow'].mean()
sch_f_rate = df[(df['Scholarship']==1) & (df['Gender']=='F')]['NoShow'].mean()
fig.text(0.5, -0.04,
    f'Insight: Chronic condition patients are the most reliable attenders '
    f'(hypertension: {hyp_rate*100:.1f}%), while Scholarship recipients show '
    f'elevated risk ({sch_f_rate*100:.1f}%), driven by structural access barriers.',
    ha='center', va='top', fontsize=9, style='italic', color='#1F3864',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#D9E8FB', edgecolor='#2E75B6'))

plt.tight_layout()
plt.savefig('fig5_5_chronic_scholarship.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 8 — EDA: Geographic Distribution & K-Means (Figure 5.6)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Patch

# ── Neighbourhood profile ────────────────────────────────────────────────────
nbhd = df.groupby('Neighbourhood').agg(
    NoShowRate  = ('NoShow',      'mean'),
    AvgWaitDays = ('WaitDays',    'mean'),
    ScholarRate = ('Scholarship', 'mean'),
    N           = ('NoShow',      'count')
).reset_index()

# ── Chi-Square: neighbourhood vs no-show ────────────────────────────────────
ct_n = pd.crosstab(df['Neighbourhood'], df['NoShow'])
chi2_n, p_n, _, _ = chi2_contingency(ct_n)
print(f'Neighbourhood Chi-Square: chi2={chi2_n:.1f}, p={p_n:.2e}')
print(f'Decision: {"Reject H₀ — significant geographic variation" if p_n < 0.05 else "Fail to reject H₀"}')

# ── K-Means clustering (k=4) ──────────────────────────────────────────────
X_n = StandardScaler().fit_transform(
    nbhd[['NoShowRate', 'AvgWaitDays', 'ScholarRate']])

km = KMeans(n_clusters=4, random_state=42, n_init=10)
nbhd['Cluster'] = km.fit_predict(X_n)

print('\nCluster profiles (mean values):')
print(nbhd.groupby('Cluster')[['NoShowRate', 'AvgWaitDays', 'ScholarRate']]
         .mean().round(3).to_string())

print('\nTop 15 highest no-show neighbourhoods:')
top15 = nbhd.nlargest(15, 'NoShowRate')[['Neighbourhood', 'NoShowRate', 'Cluster']]
top15['NoShowRate'] = (top15['NoShowRate'] * 100).round(1).astype(str) + '%'
print(top15.to_string(index=False))

# ── Figure 5.6 ───────────────────────────────────────────────────────────────
top15_df = nbhd.nlargest(15, 'NoShowRate').reset_index(drop=True)
bar_colors = []
for r in top15_df['NoShowRate']:
    if r > 0.28:   bar_colors.append(RED)
    elif r > 0.26: bar_colors.append(ORANGE)
    else:          bar_colors.append(BLUE)

cluster_colors = {0: '#C00000', 1: '#E36C09', 2: '#375623', 3: '#2E75B6'}
# Identify which cluster is highest risk
cluster_means = nbhd.groupby('Cluster')['NoShowRate'].mean()
cluster_label_map = {
    cluster_means.idxmax(): 'Cluster 1: High Risk',
}
for c in cluster_means.index:
    if c not in cluster_label_map:
        rank = sorted(cluster_means.values, reverse=True).index(cluster_means[c]) + 1
        cluster_label_map[c] = f'Cluster {rank}: ' + (['High Risk','Moderate','Low Risk','Variable'][rank-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Figure 5.6 — Geographic Distribution of No-Show Rates by Neighbourhood',
             fontsize=13, fontweight='bold', color=BLUE, y=1.01)

y_pos = np.arange(len(top15_df))
ax1.barh(y_pos, top15_df['NoShowRate'] * 100, color=bar_colors,
         edgecolor='white', height=0.7)
for i, r in enumerate(top15_df['NoShowRate']):
    ax1.text(r * 100 + 0.3, i, f'{r*100:.1f}%', va='center',
             fontsize=8.5, fontweight='bold')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top15_df['Neighbourhood'], fontsize=8.5)
ax1.axvline(BASELINE * 100, color='grey', linewidth=1.5, linestyle='--')
ax1.set_xlabel('No-Show Rate (%)', fontsize=10)
ax1.set_title('Top 15 Highest-Risk Neighbourhoods', fontsize=11, color=BLUE)
ax1.set_xlim(0, top15_df['NoShowRate'].max() * 100 * 1.2)
ax1.spines[['top', 'right']].set_visible(False)
legend_elements = [Patch(facecolor=RED,    label='>28% (Very High)'),
                   Patch(facecolor=ORANGE, label='26-28% (High)'),
                   Patch(facecolor=BLUE,   label='<26% (Elevated)')]
ax1.legend(handles=legend_elements, fontsize=8, loc='lower right')

# K-Means scatter (actual cluster data)
for c in sorted(nbhd['Cluster'].unique()):
    sub = nbhd[nbhd['Cluster'] == c]
    label = cluster_label_map.get(c, f'Cluster {c}')
    ax2.scatter(sub['AvgWaitDays'], sub['NoShowRate'] * 100,
                c=cluster_colors[c], s=50 + sub['ScholarRate'] * 300,
                alpha=0.75, label=label, edgecolors='white', linewidth=0.5)
ax2.axhline(BASELINE * 100, color='grey', linewidth=1.5, linestyle='--',
            label=f'Baseline ({BASELINE*100:.2f}%)')
ax2.set_xlabel('Average WaitDays', fontsize=10)
ax2.set_ylabel('No-Show Rate (%)', fontsize=10)
ax2.set_title('K-Means Clusters: 81 Neighbourhoods\n(bubble size ∝ Scholarship rate)',
              fontsize=11, color=BLUE)
ax2.legend(fontsize=8, loc='upper left')
ax2.spines[['top', 'right']].set_visible(False)

top10_pct = top15_df.head(10)['N'].sum() / len(df) * 100
fig.text(0.5, -0.04,
    f'Insight: Geographic concentration is stark — the top 10 high-risk neighbourhoods '
    f'account for {top10_pct:.1f}% of appointments. '
    f'Chi-Square p={p_n:.2e} confirms significant geographic variation.',
    ha='center', va='top', fontsize=9, style='italic', color='#1F3864',
    bbox=dict(boxstyle='round,pad=0.6', facecolor='#D9E8FB', edgecolor='#2E75B6'))

plt.tight_layout()
plt.savefig('fig5_6_geographic.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 9 — RQ1: Patient Risk Profiling

Chi-Square tests + Logistic Regression odds ratios + Random Forest + XGBoost + SHAP.

In [ ]:
from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.metrics       import roc_auc_score
import xgboost as xgb
import shap

# ── Chi-Square tests for all categorical predictors ──────────────────────────
print('── Chi-Square Tests ──')
cat_vars = ['Gender_enc', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism']
for v in cat_vars:
    ct = pd.crosstab(df[v], df['NoShow'])
    chi2, p, _, _ = chi2_contingency(ct)
    sig = '*** significant' if p < 0.05 else 'not significant'
    print(f'  {v:15s}: chi2={chi2:8.1f},  p={p:.4f}  → {sig}')

# ── Logistic Regression: odds ratios ────────────────────────────────────────
print('\n── Logistic Regression Odds Ratios ──')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_res, y_res)
odds_ratios = pd.DataFrame({
    'Feature': FEATURES,
    'OddsRatio': np.exp(lr.coef_[0])
}).sort_values('OddsRatio', ascending=False)
print(odds_ratios.to_string(index=False))

# ── Random Forest ────────────────────────────────────────────────────────────
print('\n── Random Forest ──')
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_res, y_res)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'Random Forest AUC: {rf_auc:.4f}')

# ── XGBoost ──────────────────────────────────────────────────────────────────
print('\n── XGBoost ──')
xgb_m = xgb.XGBClassifier(scale_pos_weight=4, eval_metric='logloss', random_state=42)
xgb_m.fit(X_res, y_res)
xgb_auc = roc_auc_score(y_test, xgb_m.predict_proba(X_test)[:, 1])
print(f'XGBoost AUC: {xgb_auc:.4f}')

# ── SHAP summary plot ─────────────────────────────────────────────────────────
print('\n── SHAP Feature Importance ──')
explainer   = shap.TreeExplainer(xgb_m)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False)
plt.title('SHAP Feature Importance — XGBoost (RQ1)', fontsize=12, color=BLUE)
plt.tight_layout()
plt.savefig('fig_rq1_shap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 10 — RQ3: SHAP Dependence Plot (SMS × WaitDays)

In [ ]:
# shap_values and xgb_m must exist from Cell 9
# (Run Cell 9 before this cell)

print('── SHAP Dependence: SMS_received × WaitDays ──')
plt.figure(figsize=(8, 5))
shap.dependence_plot(
    'SMS_received', shap_values, X_test,
    interaction_index='WaitDays',
    feature_names=FEATURES, show=False
)
plt.title('SHAP Dependence — SMS Effect moderated by WaitDays (RQ3)',
          fontsize=11, color=BLUE)
plt.tight_layout()
plt.savefig('fig_rq3_shap_dependence.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Interpretation: Points below 0 = SMS reduces no-show risk; above 0 = increases risk.')
print('Colour encodes WaitDays: short-wait (blue) benefits from SMS; long-wait (red) does not.')

---
## Cell 11 — Model Benchmarking: ROC Comparison, Cross-Validation & Fairness Audit

In [ ]:
from sklearn.metrics         import (roc_auc_score, f1_score, recall_score,
                                      precision_score, roc_curve)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree            import DecisionTreeClassifier as DTC
from fairlearn.metrics       import demographic_parity_difference

# ── Define models ─────────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DTC(max_depth=6, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200,
                               class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost'            : xgb.XGBClassifier(scale_pos_weight=4,
                               eval_metric='logloss', random_state=42),
}

# ── Benchmark ─────────────────────────────────────────────────────────────────
results = {}
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    model.fit(X_res, y_res)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'AUC'      : roc_auc_score(y_test, y_prob),
        'F1'       : f1_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
    }
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, linewidth=2,
            label=f"{name} (AUC={results[name]['AUC']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve Comparison — All Models', fontsize=12, color=BLUE)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_roc_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ── Results summary table ─────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.round(4)
print('\n── Model Performance Summary ──')
print(results_df.to_string())

# ── 5-Fold cross-validation ──────────────────────────────────────────────────
print('\n── 5-Fold Stratified Cross-Validation AUC ──')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    auc_cv = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'  {name:22s}: {auc_cv.mean():.3f} ± {auc_cv.std():.3f}')

# ── Fairness audit: Gender ────────────────────────────────────────────────────
print('\n── Gender Fairness Audit (Demographic Parity Difference) ──')
gender = X_test['Gender_enc']
for name, model in models.items():
    dpd = demographic_parity_difference(
        y_test, model.predict(X_test), sensitive_features=gender)
    flag = '✓ PASS' if abs(dpd) < 0.05 else '✗ FAIL'
    print(f'  {name:22s}: DPD = {dpd:+.4f}  {flag} (threshold < 0.05)')

---
## Cell 12 — Final Tuned XGBoost Model (Target: AUC > 0.75)

Enhanced feature engineering + tuned hyperparameters + early stopping.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# ── Feature engineering ───────────────────────────────────────────────────────
df['LogWaitDays']    = np.log1p(df['WaitDays'])
df['IsLongWait']     = (df['WaitDays'] > 7).astype(int)
df['IsVeryLongWait'] = (df['WaitDays'] > 15).astype(int)
df['AgeGroup']       = pd.cut(df['Age'], bins=[0,18,40,60,100],
                               labels=[0,1,2,3]).astype(int)

# Interaction features
df['Wait_SMS']     = df['WaitDays'] * df['SMS_received']
df['Age_Hyper']    = df['Age']     * df['Hipertension']
df['Wait_Scholar'] = df['WaitDays'] * df['Scholarship']

FEATURES_V2 = [
    'Age', 'AgeGroup', 'Scholarship', 'Hipertension', 'Diabetes',
    'Handcap', 'SMS_received', 'WaitDays', 'LogWaitDays',
    'IsLongWait', 'IsVeryLongWait',
    'Wait_SMS', 'Age_Hyper', 'Wait_Scholar',
    'Nbhd_enc'
]

X2 = df[FEATURES_V2]
y2 = df['NoShow']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42)

scale_pos_weight = (len(y2_train) - sum(y2_train)) / sum(y2_train)
print(f'scale_pos_weight (class imbalance ratio): {scale_pos_weight:.2f}')

# ── Tuned XGBoost ─────────────────────────────────────────────────────────────
final_model = xgb.XGBClassifier(
    n_estimators     = 800,
    max_depth        = 6,
    learning_rate    = 0.03,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    gamma            = 1,
    min_child_weight = 3,
    reg_alpha        = 0.5,
    reg_lambda       = 1,
    scale_pos_weight = scale_pos_weight,
    eval_metric      = 'auc',
    random_state     = 42
)

final_model.fit(
    X2_train, y2_train,
    eval_set=[(X2_test, y2_test)],
    verbose=False
)

# ── Evaluation ────────────────────────────────────────────────────────────────
y_prob_final = final_model.predict_proba(X2_test)[:, 1]
y_pred_final = final_model.predict(X2_test)
final_auc    = roc_auc_score(y2_test, y_prob_final)

print(f'\n🎯 FINAL MODEL AUC: {final_auc:.4f}')
print(f'   Target was AUC > 0.75: {"✓ ACHIEVED" if final_auc > 0.75 else "✗ NOT achieved"}')
print('\nClassification Report:')
print(classification_report(y2_test, y_pred_final,
                             target_names=['Show', 'No-Show']))

# ── SHAP for final model ──────────────────────────────────────────────────────
explainer_final   = shap.TreeExplainer(final_model)
shap_vals_final   = explainer_final.shap_values(X2_test)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals_final, X2_test, feature_names=FEATURES_V2, show=False)
plt.title(f'SHAP Feature Importance — Final Tuned XGBoost (AUC={final_auc:.3f})',
          fontsize=11, color=BLUE)
plt.tight_layout()
plt.savefig('fig_final_model_shap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Cell 13 — Results Summary Table

Compile all actual results into a single printable summary matching Tables 7.1–7.5 in the report.

In [ ]:
print('='*65)
print('  ACTUAL RESULTS SUMMARY — Replace "Expected" tables in report')
print('='*65)

print('\n── Table 7.1: Actual Logistic Regression Odds Ratios ──')
print(odds_ratios.to_string(index=False))

print('\n── Table 7.2: Actual No-Show Rate by WaitDays Bucket ──')
actual_rates = df.groupby('WaitBucket', observed=True)['NoShow'].agg(['mean','count'])
actual_rates.columns = ['NoShowRate', 'N']
actual_rates['NoShowRate'] = (actual_rates['NoShowRate']*100).round(1).astype(str) + '%'
print(actual_rates.to_string())

print('\n── Table 7.3: Actual SMS Stratified Rates ──')
sms_strat = df.groupby(['SMS_received','ShortWait'])['NoShow'].agg(['mean','count'])
sms_strat.columns = ['NoShowRate','N']
sms_strat['NoShowRate'] = (sms_strat['NoShowRate']*100).round(1).astype(str) + '%'
print(sms_strat.to_string())

print('\n── Table 7.4: Actual K-Means Cluster Profiles ──')
cluster_summary = nbhd.groupby('Cluster')[['NoShowRate','AvgWaitDays','ScholarRate','N']].mean().round(3)
print(cluster_summary.to_string())

print('\n── Table 7.5: Actual Model Performance ──')
print(results_df.to_string())
print(f'\nFinal Tuned XGBoost AUC: {final_auc:.4f}')

print('\n' + '='*65)
print('  All charts saved as PNG files in the current folder.')
print('='*65)